Here I'm going to clean the data of ISTAT accidents and SITUAS municipalities and join them in one single table.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/istat_road_accidents.csv", dtype = {"REF_AREA": str})
df_situas = pd.read_csv("../data/raw/situas_municipalities_2024.csv", dtype = {"PRO_COM_T": str, "COD_REG": str})

In [2]:
df.isnull().sum()

DATAFLOW                 0
FREQ                     0
REF_AREA                 0
DATA_TYPE                0
RESULT                   0
TIME_PERIOD              0
OBS_VALUE                0
OBS_STATUS          573552
NOTE_DS             573552
NOTE_REF_AREA       573552
NOTE_DATA_TYPE      573552
NOTE_RESULT         573552
NOTE_TIME_PERIOD    573552
BASE_PER            573552
UNIT_MEAS           573552
UNIT_MULT           573552
dtype: int64

In [3]:
df.nunique()

DATAFLOW               1
FREQ                   1
REF_AREA            8578
DATA_TYPE              2
RESULT                 3
TIME_PERIOD           24
OBS_VALUE           1830
OBS_STATUS             0
NOTE_DS                0
NOTE_REF_AREA          0
NOTE_DATA_TYPE         0
NOTE_RESULT            0
NOTE_TIME_PERIOD       0
BASE_PER               0
UNIT_MEAS              0
UNIT_MULT              0
dtype: int64

In [4]:
df = df.drop(columns=["OBS_STATUS", "NOTE_DS", "NOTE_REF_AREA", "NOTE_DATA_TYPE", "NOTE_RESULT", "NOTE_TIME_PERIOD", "BASE_PER", "UNIT_MEAS", "UNIT_MULT"])
df.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
0,IT1:41_983(1.0),A,001001,KILLINJ,F,2001,10
1,IT1:41_983(1.0),A,001001,KILLINJ,F,2002,10
2,IT1:41_983(1.0),A,001001,KILLINJ,F,2003,7
3,IT1:41_983(1.0),A,001001,KILLINJ,F,2004,13
4,IT1:41_983(1.0),A,001001,KILLINJ,F,2005,2


In [5]:
df = df[["REF_AREA", "DATA_TYPE", "RESULT", "TIME_PERIOD", "OBS_VALUE"]]
df.head()

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
0,001001,KILLINJ,F,2001,10
1,001001,KILLINJ,F,2002,10
2,001001,KILLINJ,F,2003,7
3,001001,KILLINJ,F,2004,13
4,001001,KILLINJ,F,2005,2


In [6]:
df["indicator"] = df["DATA_TYPE"] + "_" + df["RESULT"]
df["indicator"] = df["indicator"].map({"ROADACC_9": "accidents", "KILLINJ_M": "deaths", "KILLINJ_F": "injured"})
df["indicator"].value_counts()                           

indicator
injured      191184
deaths       191184
accidents    191184
Name: count, dtype: int64

In [7]:
df_pivot = df.pivot(index=["REF_AREA", "TIME_PERIOD"], columns="indicator", values="OBS_VALUE")
df_pivot = df_pivot.reset_index()
df_pivot = df_pivot.rename(columns={"REF_AREA": "municipality_code", "TIME_PERIOD": "year"})
df_pivot.head()

indicator,municipality_code,year,accidents,deaths,injured
0,001001,2001,5,0,10
1,001001,2002,5,0,10
2,001001,2003,4,0,7
3,001001,2004,9,0,13
4,001001,2005,2,0,2


Checking

In [8]:
df_pivot.isnull().sum()

indicator
municipality_code    0
year                 0
accidents            0
deaths               0
injured              0
dtype: int64

In [9]:
df_pivot.duplicated(subset= ["municipality_code", "year"]).sum()

np.int64(0)

In [10]:
df_pivot.describe()

indicator,year,accidents,deaths,injured
count,191184.000000,191184.000000,191184.000000,191184.000000
mean,2012.340541,24.997599,0.533225,35.091985
std,6.859888,257.519973,2.783505,338.109881
min,2001.000000,0.000000,0.000000,0.000000
25%,2006.000000,1.000000,0.000000,1.000000
50%,2012.000000,3.000000,0.000000,5.000000
75%,2018.000000,12.000000,0.000000,18.000000
max,2024.000000,23135.000000,363.000000,30254.000000


In [11]:
df_pivot.sort_values("accidents", ascending=False).head()

indicator,municipality_code,year,accidents,deaths,injured
116198,058091,2004,23135,260,30254
116195,058091,2001,22220,305,27865
116199,058091,2005,21902,237,28653
116200,058091,2006,21452,231,28209
116196,058091,2002,21330,363,26696


In [12]:
df_situas[df_situas["PRO_COM_T"] == "058091"]

,COD_RIP,COD_REG,COD_PROV_STORICO,COD_UTS,PRO_COM_T,PRO_COM,COMUNE,SIGLA_AUTOMOBILISTICA,CC_UTS,CC_REG,POP_LEG,ANNO_CENSIMENTO,AREA_KMQ,ANNO_AREA,POP_RES,ANNO_POP_RES,COMUNE_A
4719,3,12,58,258,058091,58091,Roma,RM,1,1,2749031,2021,1288.1875,2024,2747290,2024,NaN


I want to see how many municipalities are there each year.

In [13]:
df_pivot.groupby("year")["municipality_code"].count()

year
2001    8101
2002    8102
2003    8102
2004    8100
2005    8101
2006    8101
2007    8101
2008    8101
2009    8100
2010    8094
2011    8092
2012    8092
2013    8092
2014    8059
2015    8047
2016    8000
2017    7978
2018    7954
2019    7915
2020    7904
2021    7904
2022    7904
2023    7901
2024    6339
Name: municipality_code, dtype: int64

I notice that there are always 7900-8100 municipalities until 2023, while in 2024 only 6339. 
I want to check wether the 2024 data is incomplete or they disappeared beacuse they merged with others perhaps.

In [14]:
len(df_situas)

7896

So in 2024 there were 7896 municipalities, not 6339.

In [15]:
df_2023 = df_pivot[df_pivot["year"] == 2023]
df_2024 = df_pivot[df_pivot["year"] == 2024]

In [16]:
print("2023-municipalities with 0 accidents:", (df_2023["accidents"] == 0).sum())
print("2024-municipalities with 0 accidents:", (df_2024["accidents"] == 0).sum())

2023-municipalities with 0 accidents: 1643
2024-municipalities with 0 accidents: 0


It seems strange that in 2023 there are 1600 municipalities without accidents and 0 in 2024. Now I want to see if the total of 2024 accidents is complete.

In [17]:
print("2023 total accidents:", df_2023["accidents"].sum())
print("2024 total accidents:", df_2024["accidents"].sum())

2023 total accidents: 166525
2024 total accidents: 173364


I checked the total on Google the number of accidents in 2024 and it's the same, so the 2024 data is complete. My conclusion is that ISTAT doesn't include the municipalities with 0 accidents. So if a municipality in 2024 is missing, that means there are 0 accidents.

In [18]:
smallest_mun = df_pivot[(df_pivot["municipality_code"] == "097055") & (df_pivot["year"] >=2022)]
print(smallest_mun)

indicator municipality_code  year  accidents  deaths  injured
181067               097055  2022          0       0        0
181068               097055  2023          1       0        1


In [19]:
largest_mun = df_pivot[(df_pivot["municipality_code"] == "058091") & (df_pivot["year"] >= 2022)]
print(largest_mun)

indicator municipality_code  year  accidents  deaths  injured
116216               058091  2022      13181     150    16618
116217               058091  2023      12817     154    16023
116218               058091  2024      13924     134    17196


Comparing the smallest with the largest municipality, I can see that Rome has all 3 rows (2022, 2023, 2024) while Morterone only 2 (2022, 2023).

In [20]:
print("Morterone:", smallest_mun["accidents"].mean(), smallest_mun["accidents"].sum()/3)

Morterone: 0.5 0.3333333333333333


In [21]:
print("Rome:", largest_mun["accidents"].mean(), largest_mun["accidents"].sum()/3)

Rome: 13307.333333333334 13307.333333333334


In the next steps I want to know the average of accidents of every municipality in the last years. But since there are municipalities with 0 accidents not included, then I think it's better to use sum divided by number of years rather than mean(). Monterone doesn't have the same results indeed, while Rome does.

Cleaning the SITUAS data

In [22]:
df_situas.columns

Index(['COD_RIP', 'COD_REG', 'COD_PROV_STORICO', 'COD_UTS', 'PRO_COM_T',
       'PRO_COM', 'COMUNE', 'SIGLA_AUTOMOBILISTICA', 'CC_UTS', 'CC_REG',
       'POP_LEG', 'ANNO_CENSIMENTO', 'AREA_KMQ', 'ANNO_AREA', 'POP_RES',
       'ANNO_POP_RES', 'COMUNE_A'],
      dtype='str')

Renaming the columns in english

In [23]:
df_situas = df_situas [["PRO_COM_T", "COMUNE", "SIGLA_AUTOMOBILISTICA", "COD_REG", "COD_RIP", "CC_UTS", "POP_RES", "AREA_KMQ"]]
df_situas = df_situas.rename(columns={
    "PRO_COM_T": "municipality_code",
    "COMUNE": "municipality",
    "SIGLA_AUTOMOBILISTICA": "province",
    "COD_REG": "region_code",
    "COD_RIP": "geo_area_code",
    "CC_UTS": "province_capital",
    "POP_RES": "population",
    "AREA_KMQ": "area_km2"
})
df_situas.head()

,municipality_code,municipality,province,region_code,geo_area_code,province_capital,population,area_km2
0,001001,Agliè,TO,01,1,0,2585,13.1463
1,001002,Airasca,TO,01,1,0,3695,15.7393
2,001003,Ala di Stura,TO,01,1,0,463,46.3316
3,001004,Albiano d'Ivrea,TO,01,1,0,1624,11.7397
4,001006,Almese,TO,01,1,0,6297,17.8741


In [24]:
df_situas["geo_area_code"].unique()

array([1, 2, 3, 4, 5])

In [25]:
df_situas["region_code"].unique()

<StringArray>
['01', '02', '07', '03', '04', '05', '06', '08', '11', '09', '10', '12', '15',
 '13', '14', '16', '17', '18', '19', '20']
Length: 20, dtype: str

Translating codes in names

In [26]:
regions = {
    "01": "Piemonte", "02": "Valle D'Aosta", "03": "Lombardia", "04": "Trentino Alto Adige", "05": "Veneto", 
    "06": "Friuli Venezia Giulia", "07": "Liguria", "08": "Emilia Romagna", "09": "Toscana", "10": "Umbria", 
    "11": "Marche", "12": "Lazio", "13": "Abruzzo", "14": "Molise", "15": "Campania", 
    "16": "Puglia", "17": "Basilicata", "18": "Calabria", "19": "Sicilia", "20": "Sardegna"}

areas = {1:"North-West", 2:"North-East", 3:"Centre", 4:"South", 5:"Islands"}

df_situas["region"] = df_situas["region_code"].map(regions)
df_situas["geographic_area"] = df_situas["geo_area_code"].map(areas)

In [27]:
df_situas[["region", "geographic_area"]].isnull().sum()

region             0
geographic_area    0
dtype: int64

In [28]:
df_situas["geographic_area"].value_counts()

geographic_area
North-West    2990
South         1783
North-East    1387
Centre         968
Islands        768
Name: count, dtype: int64

In [29]:
df_situas.isnull().sum()

municipality_code     0
municipality          1
province             92
region_code           0
geo_area_code         0
province_capital      0
population            0
area_km2              0
region                0
geographic_area       0
dtype: int64

In [30]:
df_situas["municipality_code"].duplicated().sum()

np.int64(0)

In [31]:
df_situas.describe()

,geo_area_code,province_capital,population,area_km2
count,7896.000000,7896.000000,7.896000e+03,7896.000000
mean,2.487335,0.014184,7.464978e+03,38.261090
std,1.429214,0.118258,4.154247e+04,50.823422
min,1.000000,0.000000,3.200000e+01,0.122200
25%,1.000000,0.000000,9.690000e+02,11.470625
50%,2.000000,0.000000,2.383000e+03,22.456900
75%,4.000000,0.000000,6.239250e+03,44.756775
max,5.000000,1.000000,2.747290e+06,1288.187500


In [32]:
df_situas.sort_values("population").head()

,municipality_code,municipality,province,region_code,geo_area_code,province_capital,population,area_km2,region,geographic_area
7408,097055,Morterone,LC,03,1,0,32,13.6172,Lombardia,North-West
1668,014047,Pedesina,SO,03,1,0,37,6.2997,Lombardia,North-West
511,004031,Briga Alta,CN,01,1,0,40,52.1796,Piemonte,North-West
590,004112,Macra,CN,01,1,0,42,24.6562,Piemonte,North-West
119,001121,Ingria,TO,01,1,0,44,14.7505,Piemonte,North-West


In [33]:
df_situas.sort_values("population", ascending=False).head()

,municipality_code,municipality,province,region_code,geo_area_code,province_capital,population,area_km2,region,geographic_area
4719,058091,Roma,RM,12,3,1,2747290,1288.1875,Lazio,Centre
1771,015146,Milano,MI,03,1,1,1365698,181.8450,Lombardia,North-West
5104,063049,Napoli,NaN,15,4,1,909013,119.2423,Campania,South
267,001272,Torino,TO,01,1,1,853196,129.9988,Piemonte,North-West
6595,082053,Palermo,PA,19,5,1,628693,160.6676,Sicilia,Islands


I see that there are no missing values or duplicated municipalities. Also population goes from 32 (Morterone) to 2.7 milion (Rome). 

Joining ISTAT and SITUAS

In [35]:
df_join = df_pivot.merge(df_situas, on="municipality_code", how="left")
df_join.head()

,municipality_code,year,accidents,deaths,injured,municipality,province,region_code,geo_area_code,province_capital,population,area_km2,region,geographic_area
0,001001,2001,5,0,10,Agliè,TO,01,1.0,0.0,2585.0,13.1463,Piemonte,North-West
1,001001,2002,5,0,10,Agliè,TO,01,1.0,0.0,2585.0,13.1463,Piemonte,North-West
2,001001,2003,4,0,7,Agliè,TO,01,1.0,0.0,2585.0,13.1463,Piemonte,North-West
3,001001,2004,9,0,13,Agliè,TO,01,1.0,0.0,2585.0,13.1463,Piemonte,North-West
4,001001,2005,2,0,2,Agliè,TO,01,1.0,0.0,2585.0,13.1463,Piemonte,North-West


In [36]:
df_join.isnull().sum()


municipality_code        0
year                     0
accidents                0
deaths                   0
injured                  0
municipality          8639
province             10822
region_code           8615
geo_area_code         8615
province_capital      8615
population            8615
area_km2              8615
region                8615
geographic_area       8615
dtype: int64

After the joining I notice that some rows have no municipality name, population, area...these municipality codes exist in the ISTAT data but not in SITUAS. I want to know why.

In [37]:
print("Municipality codes in ISTAT:", df_pivot["municipality_code"].nunique())
print("Municipality codes in SITUAS:", df_situas["municipality_code"].nunique())

Municipality codes in ISTAT: 8578
Municipality codes in SITUAS: 7896


In [38]:
df_pivot[df_pivot["year"] < 2024].groupby("year") ["municipality_code"].count()

year
2001    8101
2002    8102
2003    8102
2004    8100
2005    8101
2006    8101
2007    8101
2008    8101
2009    8100
2010    8094
2011    8092
2012    8092
2013    8092
2014    8059
2015    8047
2016    8000
2017    7978
2018    7954
2019    7915
2020    7904
2021    7904
2022    7904
2023    7901
Name: municipality_code, dtype: int64

ISTAT has 682 more municipality codes than SITUAS. ISTAT data covers 2001-2004, while SITUAS refers to one day in 2024. The number of municipalities in ISTAT decreases every year. I think that they are municipalities that existed in the past, because the ISTAT data starts in 2001, and then don't exist anymore in 2024, because they merged with other municipalities (I searched this reason on internet, in Italy this happens when municipalities merge).

In [41]:
df_code_missing = df_pivot[df_pivot["municipality_code"].isin(df_situas["municipality_code"]) == False]
df_code_missing.head()           

indicator,municipality_code,year,accidents,deaths,injured
96,001005,2001,3,4,3
97,001005,2002,1,0,1
98,001005,2003,1,0,1
99,001005,2004,2,1,2
100,001005,2005,0,0,0


In [42]:
df_code_missing.groupby("year") ["municipality_code"].count()

year
2001    567
2002    567
2003    567
2004    565
2005    565
2006    602
2007    602
2008    602
2009    600
2010    480
2011    477
2012    477
2013    477
2014    420
2015    401
2016    327
2017    138
2018     95
2019     32
2020     17
2021     14
2022     14
2023      9
Name: municipality_code, dtype: int64

I want to know how much this loss impacts the analysis, focusing on last years (2022-2024).

In [45]:
lost_accidents = df_code_missing.groupby("year")["accidents"].sum()
total_accidents = df_pivot.groupby("year")["accidents"].sum()
(lost_accidents / total_accidents * 100).round(2)

year
2001    3.28
2002    3.20
2003    3.16
2004    3.08
2005    3.18
2006    3.59
2007    3.50
2008    3.44
2009    3.65
2010    1.18
2011    1.17
2012    1.28
2013    1.27
2014    1.00
2015    1.03
2016    0.96
2017    0.32
2018    0.20
2019    0.05
2020    0.02
2021    0.02
2022    0.02
2023    0.02
2024     NaN
Name: accidents, dtype: float64

In [43]:
print("2022 lost accidents:", df_code_missing[df_code_missing["year"] == 2022] ["accidents"].sum())
print("2023 lost accidents:", df_code_missing[df_code_missing["year"] == 2023] ["accidents"].sum())
print("2024 lost accidents:", df_code_missing[df_code_missing["year"] == 2024] ["accidents"].sum())

2022 lost accidents: 37
2023 lost accidents: 36
2024 lost accidents: 0


In recent years I lose less than 40 accidents per year out of 170.000, so very few. I can ignore them.

Joining

In [46]:
df_final = df_pivot.merge(df_situas, on="municipality_code", how="inner")
df_final.shape

(182569, 14)

In [47]:
df_final.isnull().sum()

municipality_code       0
year                    0
accidents               0
deaths                  0
injured                 0
municipality           24
province             2207
region_code             0
geo_area_code           0
province_capital        0
population              0
area_km2                0
region                  0
geographic_area         0
dtype: int64

After the joining I notice that municipality and province have missing values, I want to understand why.

In [48]:
df_final[df_final["municipality"].isnull()] [["municipality_code", "province", "year"]].head()

,municipality_code,province,year
3892,001168,TO,2001
3893,001168,TO,2002
3894,001168,TO,2003
3895,001168,TO,2004
3896,001168,TO,2005


The code is 001168, province of Torino. Now I want to know which municipalities doesn't have province.

In [49]:
df_final[df_final["province"].isnull()] ["municipality"].unique()

<StringArray>
[                   'Acerra',                  'Afragola',
                   'Agerola',                  'Anacapri',
                    'Arzano',                    'Bacoli',
           'Barano d'Ischia',                'Boscoreale',
              'Boscotrecase',                 'Brusciano',
                   'Caivano',                'Calvizzano',
                 'Camposano',                     'Capri',
         'Carbonara di Nola',                   'Cardito',
      'Casalnuovo di Napoli',              'Casamarciano',
        'Casamicciola Terme',                'Casandrino',
                'Casavatore',          'Casola di Napoli',
                   'Casoria',   'Castellammare di Stabia',
      'Castello di Cisterna',                   'Cercola',
                  'Cicciano',                  'Cimitile',
                 'Comiziano',                  'Crispano',
                     'Forio',            'Frattamaggiore',
              'Frattaminore',     'Giuglia

Now I want to see how these data are read by pandas.

In [51]:
df_check = pd.read_csv("../data/raw/situas_municipalities_2024.csv", dtype={"PRO_COM_T": str}, keep_default_na=False)
df_check[df_check["PRO_COM_T"].isin(["001168", "063049"])] [["PRO_COM_T", "COMUNE", "SIGLA_AUTOMOBILISTICA"]]

,PRO_COM_T,COMUNE,SIGLA_AUTOMOBILISTICA
164,001168,None,TO
5104,063049,Napoli,NA


Regarding missing values, I think that the reason is that Pandas reads "None" and "NA" as missing values.But "None" is a municipality province of Torino, and "NA" is province code of Napoli. To fix this I read the SITUAS file with keep_default_na=False

In [52]:
df_final["municipality"] = df_final["municipality"].fillna("None")
df_final["province"] = df_final["province"].fillna("NA")

In [53]:
df_final.isnull().sum()

municipality_code    0
year                 0
accidents            0
deaths               0
injured              0
municipality         0
province             0
region_code          0
geo_area_code        0
province_capital     0
population           0
area_km2             0
region               0
geographic_area      0
dtype: int64

In [54]:
df_final[df_final["municipality_code"].isin (["001168", "063049"])] [["municipality_code", "municipality", "province"]].drop_duplicates()

,municipality_code,municipality,province
3892,001168,None,TO
119852,063049,Napoli,NA


Now there are no missing values, and "None" and "NA" have the correct values

Saving

In [55]:
import os 
os.makedirs("../data/processed", exist_ok=True)
df_final.to_csv("../data/processed/accidents_by_municipality_year.csv", index=False)
print("Saved")

Saved
